# YZ50 — Week 4: Neural Network Language Model

- A **neural network language model** predicts the next character by using a fixed **context window** of previous characters instead of relying on a simple bigram table.

- Each character is represented by a learnable **embedding vector**, and the embeddings of the previous three characters are combined and passed through an **MLP** to predict the next character.

- The model is trained using **minibatches**, **gradient descent**, and a **train / dev / test split** to measure how well it generalizes to unseen data.

- **Tanh saturation** can cause gradients to become very small when activations approach -1 or 1. Proper **weight initialization**, such as **Kaiming initialization**, helps keep activations and gradients in a useful range.

- **BatchNorm** normalizes hidden-layer activations during training, helping stabilize the network and improve the training process.

In this week, the simple counting-based language model is replaced with a **neural network language model**, while also exploring how **embeddings, initialization, activations, and BatchNorm** affect the learning process.

# Continuing to Build Makemore: Character-Level Language Model with MLP

---

# A Neural Probabilistic Language Model — Bengio et al. (2003)

Before implementing the MLP-based character-level language model, it is useful to understand the main ideas introduced in Bengio et al. (2003), *A Neural Probabilistic Language Model*.

The paper introduces an important idea: instead of treating words as completely independent discrete symbols, we can represent them as **continuous vectors** and learn a neural network that predicts the next word from a fixed context.

This idea is closely related to what we will implement in **Makemore**, with one important difference: Bengio et al. work with **words**, while our Makemore model works with **characters**.

---

## 1. The Curse of Dimensionality

Language modeling can be viewed as learning the probability of a sequence of words:

$$
P(w_1, w_2, \dots, w_T)
$$

This can be decomposed into conditional probabilities:

$$
P(w_1, \dots, w_T)
=
\prod_{t=1}^{T}
P(w_t \mid w_1, \dots, w_{t-1})
$$

In other words, to model a sequence, we can think of predicting each next word given all the words that came before it.

The problem is that the number of possible word sequences grows **exponentially** with the vocabulary size and the sequence length.

For example, with a vocabulary of 100,000 words, the number of possible sequences of 10 words is enormous:

$$
100000^{10}
$$

Therefore, we cannot simply memorize the probability of every possible sequence.

This is the **curse of dimensionality**.

Most possible sequences will never appear in the training data. A useful language model therefore needs to **generalize** from the sequences it has seen to sequences it has never seen.

---

## 2. The Limitation of n-gram Models

A traditional solution to this problem is to use an **n-gram model**.

Instead of conditioning on the entire history, we only consider the previous $n-1$ words:

$$
P(w_t \mid w_1, \dots, w_{t-1})
\approx
P(w_t \mid w_{t-n+1}, \dots, w_{t-1})
$$

For example, a trigram model predicts the next word using only the previous two words:

$$
P(w_t \mid w_{t-2}, w_{t-1})
$$

This makes the problem more manageable, but introduces important limitations:

* The model can only use a **short context**.
* It suffers from **data sparsity**.
* Different words are treated as fundamentally different discrete symbols.
* It does not naturally understand that different words can have similar roles or meanings.

For example, consider:

```text
The cat is walking in the bedroom.
A dog was running in a room.
```

A human can recognize that:

```text
cat      ↔ dog
the      ↔ a
is       ↔ was
walking  ↔ running
bedroom  ↔ room
```

have some degree of similarity in their roles or meanings.

A traditional n-gram model does not naturally represent this kind of similarity.

---

## 3. Distributed Representations

Bengio et al. proposed representing each word using a **continuous vector**:

$$
w \rightarrow C(w) \in \mathbb{R}^m
$$

Instead of representing a word only as a discrete symbol or ID, each word gets a vector of real-valued numbers.

For example:

```text
"dog" → [ 0.12, -0.53,  0.81, ...]
"cat" → [ 0.15, -0.49,  0.77, ...]
"car" → [-0.72,  0.21,  0.34, ...]
```

These vectors are called **distributed representations** and are closely related to what we now call **embeddings**.

The important point is that these vectors are **learned from data**.

We do not manually decide:

```text
dimension 1 = animal
dimension 2 = size
dimension 3 = gender
```

Instead, the model learns numerical representations that help it perform the language modeling task.

---

## 4. From Discrete Tokens to Embeddings

To understand embeddings more clearly, let's first consider how we could represent words without them.

Suppose our vocabulary contains four words:

```text
cat   → 0
dog   → 1
car   → 2
house → 3
```

These numbers are simply **IDs**.

The fact that:

```text
cat = 0
dog = 1
car = 2
```

does not mean that `cat` is somehow closer to `dog` than to `car`.

The IDs only tell us **which token we are referring to**.

### One-Hot Encoding

One way to convert these IDs into numerical vectors is **one-hot encoding**:

```text
cat   → [1, 0, 0, 0]
dog   → [0, 1, 0, 0]
car   → [0, 0, 1, 0]
house → [0, 0, 0, 1]
```

Each vector has a length equal to the vocabulary size.

However, one-hot vectors do not encode any notion of similarity.

For example:

```text
cat → [1, 0, 0, 0]
dog → [0, 1, 0, 0]
```

and:

```text
cat   → [1, 0, 0, 0]
house → [0, 0, 0, 1]
```

are equally different in terms of their vector representation.

The model is not given any information saying that `"cat"` and `"dog"` might be more similar than `"cat"` and `"house"`.

---

## 5. The Embedding Idea

Instead of using a vocabulary-sized one-hot vector directly, we can represent each token using a much smaller **dense vector**.

For example:

```text
cat   → [ 0.21, -0.43,  0.72]
dog   → [ 0.18, -0.39,  0.68]
car   → [-0.71,  0.25,  0.11]
house → [-0.52,  0.81, -0.34]
```

These are **embeddings**.

So a useful definition is:

> **An embedding is a learnable dense vector representation of a discrete token.**

The word **learnable** is especially important.

These numbers are not manually assigned.

At the beginning of training, the embeddings can be initialized randomly:

```text
cat → [ 0.31, -0.72,  0.14]
dog → [-0.41,  0.53,  0.82]
```

During training, gradient descent updates these vectors together with the other parameters of the neural network.

The model therefore learns representations that are useful for predicting the next token.

---

## 6. The Embedding Matrix

All token embeddings can be stored in a single matrix.

Suppose:

* $V$ = vocabulary size
* $m$ = embedding dimension

Then the embedding matrix has the shape:

$$
C \in \mathbb{R}^{V \times m}
$$

For example, if we have 4 words and use a 3-dimensional embedding:

$$
C =
\begin{bmatrix}
0.21 & -0.43 & 0.72 \\
0.18 & -0.39 & 0.68 \\
-0.71 & 0.25 & 0.11 \\
-0.52 & 0.81 & -0.34
\end{bmatrix}
$$

Each row corresponds to one token:

```text
row 0 → cat
row 1 → dog
row 2 → car
row 3 → house
```

Therefore:

$$
C[0] = [0.21, -0.43, 0.72]
$$

is the embedding of `"cat"`.

Similarly:

$$
C[1] = [0.18, -0.39, 0.68]
$$

is the embedding of `"dog"`.

The operation of selecting the appropriate row is called an **embedding lookup**.

Conceptually:

```text
Token ID
   ↓
Embedding Matrix
   ↓
Select corresponding row
   ↓
Embedding Vector
```

---

## 7. Why Don't We Need to Feed One-Hot Vectors Directly?

There is an important mathematical connection between **one-hot encoding** and **embeddings**.

Suppose `"cat"` has ID `0`.

Its one-hot representation is:

```text
cat → [1, 0, 0, 0]
```

If we multiply this one-hot vector by the embedding matrix $C$:

$$
[1,0,0,0]C = C[0]
$$

The result is exactly the embedding of `"cat"`.

For `"dog"`:

$$
[0,1,0,0]C = C[1]
$$

So these two approaches are mathematically equivalent:

```text
One-hot vector
      ↓
Matrix multiplication
      ↓
Embedding vector
```

and:

```text
Token ID
   ↓
Embedding lookup
   ↓
Embedding vector
```

In practice, we use the second approach because it is much more efficient.

We do not need to create a huge one-hot vector and multiply it by the entire matrix just to select one row.

Instead, we can directly perform:

```python
C[token_id]
```

This is essentially a **lookup operation**.

---

## 8. What Does the Neural Network Actually Receive?

This distinction is important.

The neural network does not need to receive the original string:

```text
"cat"
```

and it does not necessarily need to receive the token ID:

```text
0
```

directly.

Instead, the process is:

```text
"cat"
  ↓
Token ID
  ↓
0
  ↓
Embedding lookup
  ↓
[0.21, -0.43, 0.72]
  ↓
Neural Network
```

The vector:

```text
[0.21, -0.43, 0.72]
```

is what gets passed into the neural network.

The embedding matrix itself is part of the model, so its values are updated during training.

Therefore:

$$
\text{loss}
\rightarrow
\text{backpropagation}
\rightarrow
\text{update embedding matrix}
$$

The model learns both:

1. **How to represent each token.**
2. **How to use those representations to make predictions.**

---

## 9. What Does the Embedding Actually Learn?

It is tempting to think that each dimension of an embedding represents a human-interpretable feature.

For example:

```text
dimension 1 → "animal"
dimension 2 → "size"
dimension 3 → "plurality"
```

But this is generally **not** how we should think about embeddings.

The model is not explicitly told what each dimension should represent.

Instead, it learns whatever numerical representation helps minimize the training loss.

For example, a model might learn:

```text
cat → [0.21, 0.52]
dog → [0.23, 0.49]
car → [-0.71, 0.12]
```

The fact that `"cat"` and `"dog"` are close in this space may emerge because treating them similarly helps the model make better predictions.

The individual dimensions themselves do not necessarily have clear human-interpretable meanings.

---

## 10. Why Can Embeddings Help Generalization?

This is one of the most important ideas.

Suppose two tokens have similar embeddings:

```text
x → [0.21, 0.52]
y → [0.23, 0.49]
```

The neural network therefore receives similar numerical inputs for `x` and `y`.

Because the neural network is a continuous function, similar inputs can lead to similar activations and predictions.

This allows the model to **generalize**.

Instead of learning completely independent behavior for every discrete token, the model can learn useful patterns in a continuous vector space.

This is fundamentally different from simply memorizing discrete token combinations.

---

## 11. Connecting This to Makemore

The same general idea appears in Karpathy's character-level MLP model in Makemore.

The main difference is:

> **Bengio et al. use words, while Makemore uses characters.**

Instead of:

```text
words → embeddings → MLP → next word
```

we have:

```text
characters → embeddings → MLP → next character
```

Suppose our character vocabulary is:

```text
a → 0
b → 1
c → 2
...
z → 25
. → 26
```

We can create an embedding matrix:

```python
C = torch.randn(27, 2)
```

Here:

* `27` = number of possible characters
* `2` = embedding dimension

Conceptually:

```text
a → [ 0.12,  0.54]
b → [-0.31,  0.72]
c → [ 0.81, -0.14]
...
```

Now suppose our context is:

```text
"emm"
```

and the character IDs are:

```text
e → 4
m → 12
m → 12
```

Therefore:

```text
"emm"
  ↓
[4, 12, 12]
```

We can perform an embedding lookup:

```python
emb = C[[4, 12, 12]]
```

which gives something like:

```text
[
    [0.21, 0.73],   # e
    [0.45, 0.12],   # m
    [0.45, 0.12]    # m
]
```

Each character now has its own learned vector.

---

## 12. Feeding the Embeddings into the MLP

The embeddings of the context characters are then combined into a single input vector.

For example:

```text
e → [0.21, 0.73]
m → [0.45, 0.12]
m → [0.45, 0.12]
```

Concatenating them gives:

```text
[0.21, 0.73, 0.45, 0.12, 0.45, 0.12]
```

This vector is then fed into the MLP.

The complete process is:

```text
"emm"
   ↓
Character encoding
   ↓
[4, 12, 12]
   ↓
Embedding lookup
   ↓
[
  [0.21, 0.73],
  [0.45, 0.12],
  [0.45, 0.12]
]
   ↓
Concatenate
   ↓
[0.21, 0.73, 0.45, 0.12, 0.45, 0.12]
   ↓
MLP
   ↓
Logits
   ↓
Softmax
   ↓
Probability of the next character
```

Therefore, the embedding is not the final prediction.

It is the **learned representation that transforms discrete character IDs into continuous vectors that the MLP can work with**.

---

## 13. Predicting the Next Token

The core task of the language model is to predict the next token given a fixed-size context.

At the word level:

```text
Context              Target

"The cat"       →    "is"
"cat is"        →    "walking"
"is walking"    →    "in"
```

At the character level in Makemore:

```text
Context          Target

"..."       →      "e"
"..e"       →      "m"
".em"       →      "m"
"emm"       →      "a"
```

The neural network receives the representations of the context tokens and produces a probability distribution over the vocabulary.

Mathematically:

$$
P(w_t \mid w_{t-n}, \dots, w_{t-1})
$$

for words, or:

$$
P(c_t \mid c_{t-n}, \dots, c_{t-1})
$$

for characters.

The model parameters, including the embedding matrix, are learned by maximizing the likelihood of the training data.

Equivalently, we can minimize the **negative log-likelihood**, commonly implemented as **cross-entropy loss**.

---

## 14. Connection Between Bengio and Makemore

The relationship can now be summarized as:

### Bengio et al. (2003)

```text
words
  ↓
word embeddings
  ↓
neural network
  ↓
next-word probabilities
```

### Makemore MLP

```text
characters
  ↓
character embeddings
  ↓
MLP
  ↓
next-character probabilities
```

The fundamental idea is the same:

$$
\boxed{
\text{Discrete tokens}
\rightarrow
\text{learned representations}
\rightarrow
\text{neural network}
\rightarrow
\text{next-token probabilities}
}
$$

Makemore is therefore a much smaller and simpler setting in which we can study these ideas from the ground up.

---

## 15. Why Embeddings Help Generalization

The important insight is that the model is no longer forced to treat every discrete symbol as completely unrelated.

If two tokens happen to have similar learned representations, changing one token to another results in a relatively small change in the neural network's input.

Because the MLP is a continuous function, similar inputs can produce similar activations and predictions.

This allows the model to generalize beyond the exact sequences observed during training.

In other words:

> **The model learns useful representations of the tokens together with the probability function.**

This is one of the key ideas that makes neural language models more powerful than simply memorizing n-gram counts.

---

## 16. Main Takeaways

The main ideas from Bengio et al. (2003) that are directly relevant to the Makemore MLP implementation are:

* Language modeling can be formulated as **next-token prediction**.
* The probability of a sequence can be decomposed into conditional probabilities.
* n-gram models use a limited context and suffer from data sparsity.
* Discrete tokens can be represented using **learned continuous vectors**.
* These vectors are called **distributed representations** and are closely related to what we now call **embeddings**.
* An embedding can be understood as a **learnable lookup table** that maps a token ID to a dense vector.
* One-hot encoding and embedding lookup are mathematically related: multiplying a one-hot vector by the embedding matrix selects the corresponding embedding.
* In practice, we can directly use the token ID to select the corresponding row of the embedding matrix.
* The embedding matrix is **part of the neural network parameters** and is updated through backpropagation.
* The embeddings and the neural network can be learned **jointly**.
* A neural network can use these learned representations to **generalize to unseen combinations**.
* The Makemore character-level MLP follows the same general principle, but uses **characters instead of words**.

---

## Historical Perspective

Bengio et al. (2003) can therefore be seen as an important early example of the idea:

$$
\boxed{
\text{Discrete tokens}
\rightarrow
\text{learned representations}
\rightarrow
\text{neural network}
\rightarrow
\text{next-token probabilities}
}
$$

The Makemore MLP implementation is a much smaller and simpler version of this idea.

By implementing it from scratch, we can see exactly how:

$$
\text{token IDs}
\rightarrow
\text{embeddings}
\rightarrow
\text{MLP}
\rightarrow
\text{logits}
\rightarrow
\text{probabilities}
\rightarrow
\text{loss}
$$

works internally.
